In [0]:
ls /databricks-datasets/learning-spark-v2/people/people-10m.parquet/


part-00000-04bbf354-608c-43f2-9a81-7df139d1af69.snappy.parquet*<br>
part-00001-04bbf354-608c-43f2-9a81-7df139d1af69.snappy.parquet*<br>
part-00002-04bbf354-608c-43f2-9a81-7df139d1af69.snappy.parquet*<br>
part-00003-04bbf354-608c-43f2-9a81-7df139d1af69.snappy.parquet*<br>
part-00004-04bbf354-608c-43f2-9a81-7df139d1af69.snappy.parquet*<br>
part-00005-04bbf354-608c-43f2-9a81-7df139d1af69.snappy.parquet*<br>
part-00006-04bbf354-608c-43f2-9a81-7df139d1af69.snappy.parquet*<br>
part-00007-04bbf354-608c-43f2-9a81-7df139d1af69.snappy.parquet*<br>
_SUCCESS*

In [0]:
%sql
select 
count(*)
from parquet.`/databricks-datasets/learning-spark-v2/people/people-10m.parquet/`

10000000

In [0]:
%sql
select 
*
from parquet.`/databricks-datasets/learning-spark-v2/people/people-10m.parquet/` limit 10

id	firstName	middleName	lastName	gender	birthDate	ssn	salary<br>
1	Pennie	Carry	Hirschmann	F	1955-07-02T04:00:00.000+00:00	981-43-9345	56172 <br>
2	An	Amira	Cowper	F	1992-02-08T05:00:00.000+00:00	978-97-8086	40203 <br>
3	Quyen	Marlen	Dome	F	1970-10-11T04:00:00.000+00:00	957-57-8246	53417<br>
4	Coralie	Antonina	Marshal	F	1990-04-11T04:00:00.000+00:00	963-39-4885	94727<br>
5	Terrie	Wava	Bonar	F	1980-01-16T05:00:00.000+00:00	964-49-8051	79908<br>



1.SSN Validation:<br>
a.Identify unique records based on SSN.Take last records based on birthdays<br>
b.Ensure we have correct ssn format (XXX-XX-XXXX)<br>
c.Mask the SSN <br>
2.Create a report based age group from currentdays
as example <br>
Age Group | Person Count<br>
0-20 | 100 <br>
Compute average salary based on group
visualize the result based with bar chart
3. Calculate bonus for the person who is having more than 60 years of age 
4.generate relation based 

In [0]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("People_dataset_analysis").getOrCreate()
df = spark.read.parquet("/databricks-datasets/learning-spark-v2/people/people-10m.parquet/")
display(df.show(5))

In [0]:
#checking if any null values
from pyspark.sql.functions import col
df.na.drop().count()
df.filter(col("firstName").isNull()|col("birthDate").isNull()|col("ssn").isNull()|col("ssn").isNull()|col("gender").isNull()).show()

In [0]:
%python
from pyspark.sql.functions import count, col
df.groupBy('ssn').agg(count("id").alias("SSN_Count")).where(col("SSN_Count") > 1).count()



In [0]:
df.groupBy('ssn').agg(count("id").alias("SSN_Count")).where(col("SSN_Count") > 1).show(10)

In [0]:
df.groupBy('ssn').agg(count("id").alias("SSN_Count")).where(col("SSN_Count") > 2).show(10)

In [0]:
df.filter(col("ssn")=='944-92-2756').show()

In [0]:
df.filter(col("ssn")=='972-82-2631').show()

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number,desc 
window_spec =Window.partitionBy('ssn').orderBy(desc('birthDate'))
df_distinct_ssn=df.withColumn('latest_ssn',row_number().over(window_spec)).where("latest_ssn=1")

In [0]:
df_distinct_ssn.count()

In [0]:
from pyspark.sql.functions import date_diff,current_date,round
df_new =df_distinct_ssn.withColumn('Age',round(date_diff(current_date(),col('birthDate'))/365))

In [0]:
display(df_new.show(5))

In [0]:
#check any null data is there
df_new.filter(col("gender").isNull()).show()

In [0]:
df_new.filter(col("salary").isNull()).show()

In [0]:
df_new.filter(col("birthDate").isNull()).show()

In [0]:
df_female=df_new.filter(col("gender")=='Female')
df_male=df_new.filter(col("gender")=='Male')

In [0]:
df_male.corr("Age","salary")

In [0]:
df_female.corr("Age","salary")

In [0]:
df_new.describe().show()

In [0]:
df_new.describe("salary").show()

In [0]:
df_new.filter(col("salary")> 0).describe().show()

In [0]:
from pyspark.sql.functions import abs
df_new.select(abs(col("salary"))).show(10)

In [0]:
from pyspark.sql.functions import min,max
df_new.select(min(col("salary"))).display()

In [0]:
def Salary_status(salary):
    if salary < 45000:
        return "Low"
    elif salary >= 45000 and salary < 90000:
        return "Medium"
    else:
        return "High"


In [0]:
print(Salary_status(90000))

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
salary_status_udf = udf(Salary_status,StringType())

In [0]:
df_new.withColumn("Salary_grade",salary_status_udf(df_new["salary"])).show(10)

In [0]:
def age_bining(age):
    if age < 20:
        return "Below 20"
    elif age >= 20 and age < 40:
        return "Between 20 and 40"
    elif age >= 40 and age < 60:
        return "Between 40 and 60"
    elif age >= 60 and age < 80:
        return "Between 60 and 80"
    else:
        return "Above 80"

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
age_bining_udf = udf(age_bining,StringType())

In [0]:
df_new =df_new.withColumn("age_group",age_bining_udf(df_new["age"]))
df_new.show()

In [0]:
from pyspark.sql.functions import avg,mean,mode
df_new.groupby("age_group","gender").agg(count("id").alias("Person_count"),
                                         max("salary").alias("Max_Salary"),
                                         min("salary").alias("Min_Salary"),
                                         mode("salary").alias("mode_Salary"),
                                         avg("salary").alias("Avg_Salary")).show()

In [0]:
q1 = df_new.approxQuantile(
    "salary",
    [0.25],
    0.01
)[0]
q3 = df_new.approxQuantile(
    "salary",
    [0.75],
    0.01
)[0]
display(q1)
print("---------------------")
display(q3)

In [0]:
iqr= q3-q1
lower_bound=q1-iqr*1.5
upper_bound=q3+iqr*1.5
print(iqr)
print(lower_bound)
print(upper_bound)

In [0]:
df_new.filter(df["salary"] > upper_bound).count()

In [0]:
df_new.filter(df["salary"] < lower_bound).count()

In [0]:
df_new.count()

In [0]:
((31591+30887)/9405965)

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
df_pd=df_new.toPandas()

In [0]:
plt.figure(figsize=(8,6))
sns.boxplot(y=df_pd["salary"])
plt.xlabel("box plot")
plt.show()

In [0]:
plt.figure(figsize=(8,6))
sns.violinplot(y=df_pd["salary"])
plt.xlabel("box plot")
plt.show()

In [0]:
df_new.filter(df["salary"] < 0).count()

In [0]:
from pyspark.sql.functions import col,when
df_new=df_new.select(col("id"),
                     col("firstName"),
                     col("middleName"),
                     col("lastName"),
                     col("gender"),
                     col("age"),
                     col("ssn"),
                     when(df_new["salary"]<0,lower_bound).otherwise(df_new["salary"]).alias("salary")
                     )
df_new.show()

In [0]:
df_new.filter(df["salary"] < 0).count()

In [0]:
df_new.printSchema()

In [0]:
df_new.filter((df["salary"]>= lower_bound) & (df["salary"] <= upper_bound)).count()

In [0]:
df_people =df_new.filter((df["salary"]>= lower_bound) & (df["salary"] <= upper_bound))

In [0]:
df_people.show(5)

In [0]:
from pyspark.sql.functions import expr
df_people.withColumn('SSN_CHECK',expr("length(ssn)")).filter(col('SSN_CHECK') == 11 ).show()

In [0]:
%sh
pwd

In [0]:
df_people.coalesce(1)
df_people.write.parquet('/Volumes/movie_catalog/movie_schema/temp_storage/people_data')